# Solving Optimal Stopping Problems with Reinforcement Learning

This notebook contains the code used to produce the results in my thesis *Solving Optimal Stopping Problems with Reinforcement Learning*.
The central goal is to develop a unified reinforcement learning framework for several classes of optimal stopping problems in settings where the underlying dynamics are not fully known.
The key idea is to replace the discontinuous stop/continue decision with a smooth, intensity-based randomised control and to introduce controlled exploration via entropy regularisation, which yields a stable policy iteration method.
This notebook implements the core algorithm for single stopping, extends it via the reduction principle to double stopping, and includes the equilibrium-based method for time-inconsistent stopping.

In [ ]:
# Imports
import math
import sys
import os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt
import glob
import re
import copy
from collections import defaultdict

plt.rcParams.update({'font.size': 14})
COLOURS = ['#3366cc', '#66cc33', '#ffcc33']

## Block 1: Plot of Shannon Entropy Curve $H$
This cell plots three distributions with varying Shannon differential entropy $H(\pi)=-\int_U\pi(u)\log\pi(u)du$.

In [ ]:
# Block 1 - Shannon Entropy H
plt.style.use('seaborn-v0_8-white')

u = np.linspace(0, 3, 1000)

uniform = np.ones_like(u) / np.trapz(np.ones_like(u), u)

normal = np.exp(-0.5 * ((u - 1.5)/0.4)**2)
normal /= np.trapz(normal, u)

sharp = np.exp(-0.5 * ((u - 1.5)/0.1)**2)
sharp /= np.trapz(sharp, u)

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(u, sharp, linewidth=2, color=COLOURS[0], label='Small $H$')
ax.plot(u, normal, linewidth=2, color=COLOURS[1], label='Medium $H$')
ax.plot(u, uniform, linewidth=2, color=COLOURS[2], label='Large $H$')


ax.spines['left'].set_position('zero')
ax.spines['bottom'].set_position('zero')
ax.spines['right'].set_color('none')
ax.spines['top'].set_color('none')

ax.xaxis.set_ticks_position('bottom')
ax.yaxis.set_ticks_position('left')

ax.set_xticks([])
ax.set_yticks([])

ax.set_xlabel('')
ax.set_ylabel('')

ax.text(
    1.02, 0, r'$u$',
    transform=ax.get_yaxis_transform(),
    fontsize=12,
    ha='left', va='center'
)

ax.text(
    0, 1.02, r'$\pi(u)$',
    transform=ax.get_xaxis_transform(),
    fontsize=12,
    ha='center', va='bottom'
)

ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("Plots/entropy_distributions.png")
plt.show()

## Block 2: Plotting $\pi^*$ for varying $\lambda$
This cell the softmax/entropy‑regularised optimal policy $\pi^*(u)$ for multiple $\lambda$ to show how exploration smooths the control distribution.

In [ ]:
# Block 2 — Softmax policy pi^* vs lambda

plt.style.use('seaborn-v0_8-white')

u = np.linspace(-1, 5, 10000)

def Phi(u):
    return -(u - 2)**2 + 1

def softmax(u, lam):
    w = np.exp(Phi(u) / lam)
    return w / np.trapz(w, u)

lambdas = {"Small $\lambda$":0.05, "Medium $\lambda$":0.3, "Large $\lambda$":1}

fig, ax = plt.subplots(figsize=(6,4))

i=0
for lam in lambdas.keys():
    ax.plot(u, softmax(u, lambdas[lam]), color=COLOURS[i], linewidth=2, label=rf'{lam}')
    i += 1

ax.spines['left'].set_position('zero')
ax.spines['bottom'].set_position('zero')
ax.spines['right'].set_color('none')
ax.spines['top'].set_color('none')

ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticks([])
ax.set_yticks([])

ax.text(1.02, 0, r'$u$', transform=ax.get_yaxis_transform(),
        fontsize=12, ha='left', va='center')

ax.text(0, 1.02, r'$\pi^*(u)$', transform=ax.get_xaxis_transform(),
        fontsize=12, ha='center', va='bottom')

ax.legend(frameon=False)
ax.grid(False)

plt.tight_layout()
plt.savefig("Plots/softmax_policy.png")
plt.show()

## Block 3: Realisation of Sample Survival Process $p$

This cell simulates a time‑varying intensity $\pi_t$ and computes the corresponding survival process $p_t=\exp(-\int_0^t \pi_s ds)pt$, then plots both.

In [ ]:
# Block 3 - — Intensity pi and survival p
plt.style.use('seaborn-v0_8-white')

t = np.linspace(0, 5, 10000)

pi = 0.6 + 0.3*np.sin(2*t)

dt = t[1] - t[0]
p = np.ones_like(t)

for i in range(1, len(t)):
    p[i] = p[i-1] * np.exp(-pi[i-1]*dt)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))

ax = axs[0]
ax.plot(t, pi, color='black', linewidth=2)

ax.spines['left'].set_position('zero')
ax.spines['bottom'].set_position('zero')
ax.spines['right'].set_color('none')
ax.spines['top'].set_color('none')

ax.xaxis.set_ticks_position('bottom')
ax.yaxis.set_ticks_position('left')

ax.set_xticks([])
ax.set_yticks([0, 0.5, 1])
ax.set_yticklabels(['', r'$0.5$', r'$1$'])

ax.tick_params(direction='out', length=4, width=0.8, labelsize=10)

ax.text(1.02, 0, r'$t$', transform=ax.get_yaxis_transform(),
        fontsize=12, ha='left', va='center')

ax.text(0, 1.02, r'$\pi_t$', transform=ax.get_xaxis_transform(),
        fontsize=12, ha='center', va='bottom')


ax = axs[1]
ax.plot(t, p, color='black', linewidth=2)

ax.spines['left'].set_position('zero')
ax.spines['bottom'].set_position('zero')
ax.spines['right'].set_color('none')
ax.spines['top'].set_color('none')

ax.xaxis.set_ticks_position('bottom')
ax.yaxis.set_ticks_position('left')

ax.set_xticks([])
ax.set_yticks([1])

ax.tick_params(direction='out', length=4, width=0.8, labelsize=10)

ax.text(1.02, 0, r'$t$', transform=ax.get_yaxis_transform(),
        fontsize=12, ha='left', va='center')

ax.text(0, 1.02, r'$p_t$', transform=ax.get_xaxis_transform(),
        fontsize=12, ha='center', va='bottom')

plt.tight_layout()
plt.savefig("Plots/survival_process.png")
plt.show()

## Block 4: Plotting Unnormalised Entropy Curve $R$

This cell plots the unnormalised entropy $R(\pi)=\pi-\pi\log\pi$ used in the intensity‑regularised stopping objective.

In [ ]:
# Block 4 - Unnormalised entropy R
plt.style.use('seaborn-v0_8-white')

def R(pi):
    return pi - pi * np.log(pi)

pi = np.linspace(0.001, 5, 1000)
R_pi = R(pi)

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(pi, R_pi, color='black', linewidth=2)

ax.spines['left'].set_position('zero')
ax.spines['bottom'].set_position('zero')
ax.spines['right'].set_color('none')
ax.spines['top'].set_color('none')

ax.xaxis.set_ticks_position('bottom')
ax.yaxis.set_ticks_position('left')

x_ticks = [0, 1, 2,3,4,5]
y_ticks = [-3, -2.5, -2, -1.5, -1, -0.5, 0, 0.5, 1, 1.5]
ax.set_xticks([t for t in x_ticks if t != 0])
ax.set_yticks([t for t in y_ticks if t != 0])

ax.tick_params(direction='out', length=4, width=0.8, labelsize=10)

ax.set_xlabel('')
ax.set_ylabel('')

ax.text(
    1.02, 0, r'$\pi$',
    transform=ax.get_yaxis_transform(),
    fontsize=12,
    ha='left', va='center'
)

ax.text(
    0, 1.02, r'$R(\pi)$',
    transform=ax.get_xaxis_transform(),
    fontsize=12,
    ha='center', va='bottom'
)

ax.grid(False)
plt.tight_layout()
plt.savefig("Plots/entropy_R_curve.png")
plt.show()

## Block 5: Simulating GBM

Here we simulate paths of geometric Brownian motion for illustrative purposes.

In [ ]:
# Block 5 - Simulated GBM
S0 = 40.0
r = 0.06
sigma = 0.40
T = 1.0

n_steps = 500
dt = T / n_steps

t = np.linspace(0, T, n_steps + 1)
rng = np.random.default_rng(4)
dW = rng.normal(0.0, np.sqrt(dt), size=n_steps)
W = np.concatenate([[0.0], np.cumsum(dW)])
S = S0 * np.exp((r - 0.5 * sigma**2) * t + sigma * W)

plt.figure(figsize=(8, 4.5))
plt.plot(t, S, linewidth=2)
plt.xlabel('t')
plt.ylabel(r'$X$')
plt.grid(True, alpha=0.3)
plt.savefig("Plots/GBM_realisation.png", bbox_inches="tight", pad_inches=0.02)
plt.show()

## Block 6: Logistic Best-response Policy
This cell plots the best‑response stopping policy to show how $\lambda$ controls smoothness and decisiveness in discrete time time-inconsistent stopping.

In [ ]:
# Block 6 - Logistic best-response policy vs lambda
plt.style.use('seaborn-v0_8-white')

def pi_star(z, lam):
    return 1 / (1 + np.exp(z / lam))

z = np.linspace(-5, 5, 10000)

lambdas = [0.5, 1.0, 2.0]
labels = [r'Small $\lambda$', r'Medium $\lambda$', r'Large $\lambda$']

fig, ax = plt.subplots(figsize=(6, 4))

i=0
for lam, label in zip(lambdas, labels):
    ax.plot(z, pi_star(-z, lam), color=COLOURS[i], linewidth=2, label=label)
    i += 1


ax.spines['left'].set_position('zero')
ax.spines['bottom'].set_position('zero')
ax.spines['right'].set_color('none')
ax.spines['top'].set_color('none')

ax.xaxis.set_ticks_position('bottom')
ax.yaxis.set_ticks_position('left')

ax.set_xticks([])
ax.set_yticks([1])

ax.tick_params(direction='out', length=4, width=0.8, labelsize=10)

ax.set_xlabel('')
ax.set_ylabel('')

ax.text(1.02, 0, r'$J^{\tilde{\pi}}(x)-g(x)$',
        transform=ax.get_yaxis_transform(),
        fontsize=12, ha='left', va='center')

ax.text(0, 1.02, r'$\pi^*(x)$',
        transform=ax.get_xaxis_transform(),
        fontsize=12, ha='center', va='bottom')

ax.legend(frameon=False)

ax.grid(False)
plt.tight_layout()
plt.savefig("Plots/logistic_policy.png")
plt.show()

# Numerical Experiments

The following code blocks are used for the numerical experiments in the thesis.

## Experiment 1: American Put Option (Time Consistent)
In this block we follow Algorithm 1 to train time‑slice neural networks to approximate the intensity‑regularised value function for an American put under GBM, sweeping $\lambda$ and reporting the pricing error against the reference price.

In [ ]:
# Experiment 1 - American Put
S0 = 40.0
K  = 40.0
r  = 0.06
sigma = 0.40
T  = 1.0

L = 50
dt = T / L

iterations     = 3000
batch_size     = 2**10
test_size      = 2**18
learning_rate  = 1e-3
eval_every     = 10

seeds = [i for i in range(1,101)]
lambdas = [10.0, 1e-1, 1e-4]

reference_price = 5.311

eps = 1e-12
pi_max = (1.0 - 1e-6) / dt
LOG_PI_MIN = math.log(eps)
LOG_PI_MAX = math.log(pi_max)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Seed: {seed}")


    def simulate_gbm(N):
        S = torch.empty((N, L + 1), device=device)
        S[:, 0] = S0
        Z = torch.randn((N, L), device=device)
        drift = (r - 0.5 * sigma**2) * dt
        vol = sigma * math.sqrt(dt)
        for l in range(L):
            S[:, l+1] = S[:, l] * torch.exp(drift + vol * Z[:, l])
        return S

    def payoff_put(S):
        return torch.clamp(K - S, min=0.0)


    class TimeSliceNet(nn.Module):
        def __init__(self, width=21):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(3, width), nn.ReLU(),
                nn.Linear(width, width), nn.ReLU(),
                nn.Linear(width, 1)
            )
            # Xavier init like typical FC nets
            for m in self.net:
                if isinstance(m, nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    nn.init.zeros_(m.bias)

            # helpful to start V≈g (stable), not required but ok
            nn.init.zeros_(self.net[-1].weight)
            nn.init.zeros_(self.net[-1].bias)

        def forward(self, S, g, t_norm):
            x = torch.stack([S, g, t_norm], dim=-1)
            return g + self.net(x).squeeze(-1)


    @torch.no_grad()
    def evaluate_boundary(nets, S_test, g_test):
        disc = torch.exp(-r * torch.arange(L+1, device=device) * dt)

        payoff = torch.zeros(S_test.shape[0], device=device)
        exercised = torch.zeros(S_test.shape[0], dtype=torch.bool, device=device)

        for l in range(1,L):
            if exercised.all():
                break
            S_l = S_test[:, l]
            g_l = g_test[:, l]
            t_norm = torch.full_like(S_l, l / L)

            V_l = nets[l](S_l, g_l, t_norm)

            stop = (V_l < g_l - 1e-6) & (~exercised)
            payoff[stop] = disc[l] * g_l[stop]
            exercised[stop] = True

        payoff[~exercised] = disc[L] * g_test[~exercised, L]

        value = payoff.mean().item()
        rel_err = abs(value - reference_price) / reference_price
        return value, rel_err


    def train_all_lambdas():
        models = {
            lam: nn.ModuleList([TimeSliceNet().to(device) for _ in range(L)])
            for lam in lambdas
        }
        optims = {
            lam: [optim.Adam(net.parameters(), lr=learning_rate) for net in models[lam]]
            for lam in lambdas
        }

        with torch.no_grad():
            S_test = simulate_gbm(test_size)
            g_test = payoff_put(S_test)

        history = {lam: [] for lam in lambdas}

        for it in range(1, iterations + 1):
            S = simulate_gbm(batch_size)
            g = payoff_put(S)

            for lam in lambdas:
                nets = models[lam]
                opts = optims[lam]

                for l in reversed(range(L)):
                    S_l = S[:, l]
                    g_l = g[:, l]
                    t_norm = torch.full_like(S_l, l / L)

                    with torch.no_grad():
                        V_l_old = nets[l](S_l, g_l, t_norm)
                        log_pi = -(V_l_old - g_l) / lam
                        log_pi = torch.clamp(log_pi, LOG_PI_MIN, LOG_PI_MAX)
                        pi = torch.exp(log_pi)

                    if l == L - 1:
                        V_next = g[:, l + 1]
                    else:
                        with torch.no_grad():
                            V_next = nets[l + 1](
                                S[:, l + 1],
                                g[:, l + 1],
                                torch.full_like(S_l, (l + 1)/L)
                            )
                    R_pi = pi - pi * torch.log(pi + 1e-12)
                    V_l = nets[l](S_l, g_l, t_norm)
                    td = (
                        g_l * pi * dt
                        + math.exp(-r * dt) * (1.0 - pi * dt) * V_next
                        - V_l
                    )

                    loss = (td ** 2).mean()
                    opts[l].zero_grad()
                    loss.backward()
                    opts[l].step()

            if it == 1 or it % eval_every == 0:
                for lam in lambdas:
                    val, err = evaluate_boundary(models[lam], S_test, g_test)
                    history[lam].append((it, val, err))

                msg = " | ".join(
                    [f"λ={lam:g} val={history[lam][-1][1]:.4f} err={history[lam][-1][2]:.3e}"
                     for lam in lambdas]
                )
                print(f"\riter {it:4d}/{iterations} :: {msg}", end="")
                sys.stdout.flush()

        print()
        return {lam: np.array(hist, dtype=float) for lam, hist in history.items()}


    results = train_all_lambdas()

    for lam, hist in results.items():
        df = pd.DataFrame(hist, columns=["iteration", "value", "relative_error"])
        df.to_csv(f"Results/american_put_lambda_{lam}_seed_{seed}.csv", index=False)

In [ ]:
# Results Table + Plots

files = glob.glob(r"Results/american_put_lambda_*_seed_*.csv")
data_by_lambda = defaultdict(list)
pattern = re.compile(r"Results\\american_put_lambda_(.*)_seed_(\d+)\.csv")
for fname in files:
    match = pattern.match(fname)
    if match is None:
        continue

    lam_str, seed_str = match.groups()
    lam = float(lam_str)

    df = pd.read_csv(fname)
    data_by_lambda[lam].append(df)


# Results table
P_ref = 5.311
results = []

for lam, dfs in sorted(data_by_lambda.items()):
    final_values = [df["value"].iloc[-1] for df in dfs]

    P_hat = np.mean(final_values)
    P_sd = np.std(final_values, ddof=1)  # sample SD
    rel_error = abs(P_hat - P_ref) / P_ref

    results.append((lam, P_hat, P_sd, rel_error))

for lam, P_hat, P_sd, rel_error in results:
    print(
        f"lambda = {lam:.0e}, "
        f"Value = {P_hat:.4f}, "
        f"SD = {P_sd:.3f}, "
        f"Relative error = {rel_error:.3f}"
    )

# Learning curve
plt.figure(figsize=(8, 5))

i = 0
for lam, dfs in reversed(sorted(data_by_lambda.items())):
    rel_errors = np.stack([df["relative_error"].values for df in dfs])
    iterations = dfs[0]["iteration"].values
    mean_err = rel_errors.mean(axis=0)
    std_err = rel_errors.std(axis=0)
    plt.plot(iterations, mean_err, color=COLOURS[i], label=fr"$\lambda={lam}$")
    i += 1

plt.yscale("log")
plt.xlabel("Number of Steps")
plt.ylabel("Relative Error")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("Plots/RL_put_LR.png")
plt.show()

## Experiment 2: Double Stopping - Exchange option
In this block we follow Algorithm 2 to solve the exchange option by using the reduction principle. We solve the two inner single‑stopping problems then the stopping problem to determine the value of the contract.

In [ ]:
# Experiment 2 - Exchange option
X10 = 60.0
X20 = 60.0
r   = 0.06
sigma = 0.40
rho = 0.50
T   = 1.0

L = 50
dt = T / L

iterations_inner = 3000
iterations_outer = 3000

batch_size = 2**10
test_size  = 2**18

learning_rate = 1e-3
eval_every = 10

seeds   = [i for i in range(1, 101)]
lambdas = [10.0, 0.1, 0.0001]

eps = 1e-12
pi_max = (1.0 - 1e-6) / dt
LOG_PI_MIN = math.log(eps)
LOG_PI_MAX = math.log(pi_max)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


@torch.no_grad()
def simulate_two_gbm(N: int):
    X1 = torch.empty((N, L+1), device=device)
    X2 = torch.empty((N, L+1), device=device)
    X1[:, 0] = X10
    X2[:, 0] = X20

    Z1 = torch.randn((N, L), device=device)
    Z2 = torch.randn((N, L), device=device)
    Z2 = rho * Z1 + math.sqrt(1.0 - rho**2) * Z2

    drift = (r - 0.5 * sigma**2) * dt
    vol   = sigma * math.sqrt(dt)

    for l in range(L):
        X1[:, l+1] = X1[:, l] * torch.exp(drift + vol * Z1[:, l])
        X2[:, l+1] = X2[:, l] * torch.exp(drift + vol * Z2[:, l])

    return X1, X2


@torch.no_grad()
def simulate_one_gbm(N: int, S0: torch.Tensor, sigma_local: float):
    S = torch.empty((N, L+1), device=device)
    S[:, 0] = S0

    Z = torch.randn((N, L), device=device)
    drift = (r - 0.5 * sigma_local**2) * dt
    vol   = sigma_local * math.sqrt(dt)

    for l in range(L):
        S[:, l+1] = S[:, l] * torch.exp(drift + vol * Z[:, l])

    return S


def payoff_call_unit(z: torch.Tensor):
    return torch.clamp(z - 1.0, min=0.0)

def payoff_put_unit(z: torch.Tensor):
    return torch.clamp(1.0 - z, min=0.0)


class TimeSliceNet(nn.Module):
    def __init__(self, in_dim: int, width: int = 21):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, width), nn.ReLU(),
            nn.Linear(width, width), nn.ReLU(),
            nn.Linear(width, 1)
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x: torch.Tensor):
        return self.net(x).squeeze(-1)



class SingleStoppingIntensityRL:
    def __init__(self, lam: float, feat_dim: int, width: int = 21, lr: float = 1e-3):
        self.lam = lam
        self.feat_dim = feat_dim
        self.nets = nn.ModuleList([TimeSliceNet(in_dim=feat_dim + 2, width=width).to(device) for _ in range(L)])
        self.opts = [optim.Adam(net.parameters(), lr=lr) for net in self.nets]

    def _pi(self, V: torch.Tensor, g: torch.Tensor):
        log_pi = -(V - g) / self.lam
        log_pi = torch.clamp(log_pi, LOG_PI_MIN, LOG_PI_MAX)
        return torch.exp(log_pi)

    def V_slice(self, l: int, feat_l: torch.Tensor, g_l: torch.Tensor):
        t_norm = torch.full((feat_l.shape[0], 1), l / L, device=device)
        x = torch.cat([feat_l, g_l.unsqueeze(-1), t_norm], dim=-1)
        add = self.nets[l](x)
        return g_l + add

    def train_one_batch_backward(self, feat: torch.Tensor, g: torch.Tensor):
        disc_step = math.exp(-r * dt)
        N = g.shape[0]

        for l in reversed(range(L)):
            feat_l = feat[:, l, :]
            g_l = g[:, l]

            with torch.no_grad():
                V_old = self.V_slice(l, feat_l, g_l)
                pi = self._pi(V_old, g_l)

                if l == L-1:
                    V_next = g[:, l+1]
                else:
                    feat_next = feat[:, l+1, :]
                    g_next = g[:, l+1]
                    V_next = self.V_slice(l+1, feat_next, g_next)

            V_l = self.V_slice(l, feat_l, g_l)
            td = g_l * pi * dt + disc_step * (1.0 - pi * dt) * V_next - V_l

            loss = (td**2).mean()
            self.opts[l].zero_grad()
            loss.backward()
            self.opts[l].step()

    @torch.no_grad()
    def boundary_stop_time(self, feat: torch.Tensor, g: torch.Tensor, tol: float = 1e-6):
        N = g.shape[0]
        stop_idx = torch.full((N,), L, dtype=torch.long, device=device)

        for l in range(1, L):
            g_l = g[:, l]
            feat_l = feat[:, l, :]
            V_l = self.V_slice(l, feat_l, g_l)
            mask = (V_l <= g_l - tol) & (stop_idx == L)
            stop_idx = torch.where(mask, torch.full_like(stop_idx, l), stop_idx)

        return stop_idx

    @torch.no_grad()
    def evaluate_boundary_value(self, feat: torch.Tensor, g: torch.Tensor):
        disc = torch.exp(-r * torch.arange(L+1, device=device) * dt)
        tau = self.boundary_stop_time(feat, g)
        payoff = disc[tau] * g[torch.arange(g.shape[0], device=device), tau]
        return payoff.mean().item(), payoff.std(unbiased=False).item()


def feat_inner(z: torch.Tensor):
    return z.unsqueeze(-1)

def feat_outer(X1: torch.Tensor, X2: torch.Tensor):
    eps_ = 1e-8
    z12 = X1 / (X2 + eps_)
    z21 = X2 / (X1 + eps_)
    logz = torch.log(z12 + eps_)
    return torch.stack([X1, X2, z12, logz], dim=-1)


@torch.no_grad()
def inner_value_grid(inner: SingleStoppingIntensityRL, z: torch.Tensor, payoff_fn):
    g = payoff_fn(z)
    V = torch.empty_like(g)
    V[:, L] = g[:, L]
    for l in range(L):
        feat_l = z[:, l].unsqueeze(-1)
        V[:, l] = inner.V_slice(l, feat_l, g[:, l])
    return V, g


@torch.no_grad()
def phi_on_paths(X1: torch.Tensor, X2: torch.Tensor,
                 inner_call: SingleStoppingIntensityRL,
                 inner_put: SingleStoppingIntensityRL):
    eps_ = 1e-8
    z12 = X1 / (X2 + eps_)
    z21 = X2 / (X1 + eps_)

    Vc, _ = inner_value_grid(inner_call, z12, payoff_call_unit)
    Vp, _ = inner_value_grid(inner_put,  z21, payoff_put_unit)

    u1 = X2 * Vc
    u2 = X1 * Vp
    return torch.maximum(u1, u2), Vc, Vp


@torch.no_grad()
def evaluate_exchange_option(X1: torch.Tensor, X2: torch.Tensor,
                             inner_call: SingleStoppingIntensityRL,
                             inner_put: SingleStoppingIntensityRL,
                             outer: SingleStoppingIntensityRL,
                             tol: float = 1e-6):
    N = X1.shape[0]
    disc = torch.exp(-r * torch.arange(L+1, device=device) * dt)

    phi, Vc_ratio, Vp_ratio = phi_on_paths(X1, X2, inner_call, inner_put)
    feat_o = feat_outer(X1, X2)
    tau_theta = outer.boundary_stop_time(feat_o, phi, tol=tol)

    payoff = torch.zeros((N,), device=device)

    at_T = (tau_theta == L)
    payoff[at_T] = disc[L] * torch.clamp(X1[at_T, L] - X2[at_T, L], min=0.0)

    idx = torch.arange(N, device=device)
    not_T = ~at_T
    if not_T.any():
        th = tau_theta[not_T]  # (M,)
        idxM = idx[not_T]

        x1_th = X1[idxM, th]
        x2_th = X2[idxM, th]

        u1_th = x2_th * Vc_ratio[idxM, th]
        u2_th = x1_th * Vp_ratio[idxM, th]

        choose_call_branch = (u1_th >= u2_th)
        choose_put_branch  = ~choose_call_branch

        if choose_call_branch.any():
            idxC = idxM[choose_call_branch]
            thC  = th[choose_call_branch]
            Kc   = X2[idxC, thC]
            zC   = X1[idxC, :] / (Kc.unsqueeze(-1) + 1e-8)
            gC   = payoff_call_unit(zC)

            Vcall = torch.empty_like(gC)
            Vcall[:, L] = gC[:, L]
            for l in range(L):
                feat_l = zC[:, l].unsqueeze(-1)
                Vcall[:, l] = inner_call.V_slice(l, feat_l, gC[:, l])

            grid = torch.arange(L+1, device=device).unsqueeze(0).expand(zC.shape[0], -1)
            valid = (grid >= thC.unsqueeze(-1))
            stop_mask = (Vcall <= gC - tol) & valid
            stop_idx = torch.where(stop_mask.any(dim=1), stop_mask.float().argmax(dim=1), torch.full((zC.shape[0],), L, device=device, dtype=torch.long))
            tau1 = stop_idx

            payoff[idxC] = disc[tau1] * torch.clamp(X1[idxC, tau1] - Kc, min=0.0)

        if choose_put_branch.any():
            idxP = idxM[choose_put_branch]
            thP  = th[choose_put_branch]
            Kp   = X1[idxP, thP]
            zP   = X2[idxP, :] / (Kp.unsqueeze(-1) + 1e-8)
            gP   = payoff_put_unit(zP)

            Vput = torch.empty_like(gP)
            Vput[:, L] = gP[:, L]
            for l in range(L):
                feat_l = zP[:, l].unsqueeze(-1)
                Vput[:, l] = inner_put.V_slice(l, feat_l, gP[:, l])

            grid = torch.arange(L+1, device=device).unsqueeze(0).expand(zP.shape[0], -1)
            valid = (grid >= thP.unsqueeze(-1))
            stop_mask = (Vput <= gP - tol) & valid
            stop_idx = torch.where(stop_mask.any(dim=1), stop_mask.float().argmax(dim=1), torch.full((zP.shape[0],), L, device=device, dtype=torch.long))
            tau2 = stop_idx

            payoff[idxP] = disc[tau2] * torch.clamp(Kp - X2[idxP, tau2], min=0.0)

    return payoff.mean().item(), payoff.std(unbiased=False).item()


def train_exchange_for_seed(seed: int):
    set_seed(seed)
    print(f"\nSeed: {seed} | device={device}")

    with torch.no_grad():
        X1_test, X2_test = simulate_two_gbm(test_size)

        z0_test = torch.exp(0.25 * torch.randn((test_size,), device=device))
        Z_test = simulate_one_gbm(test_size, z0_test, sigma)

        feat_test_call = feat_inner(Z_test)
        feat_test_put  = feat_inner(Z_test)
        g_test_call = payoff_call_unit(Z_test)
        g_test_put  = payoff_put_unit(Z_test)

        phi_test_all = {}
        feat_outer_test = feat_outer(X1_test, X2_test)

    inner_call = {}
    inner_put  = {}
    outer      = {}

    hist_u1  = {lam: [] for lam in lambdas}
    hist_u2  = {lam: [] for lam in lambdas}
    hist_phi = {lam: [] for lam in lambdas}

    for lam in lambdas:
        inner_call[lam] = SingleStoppingIntensityRL(lam, feat_dim=1, lr=learning_rate)
        inner_put[lam]  = SingleStoppingIntensityRL(lam, feat_dim=1, lr=learning_rate)
        outer[lam]      = SingleStoppingIntensityRL(lam, feat_dim=4, lr=learning_rate)

    # Inner problems
    for it in range(1, iterations_inner + 1):

        with torch.no_grad():
            z0 = torch.exp(0.25 * torch.randn((batch_size,), device=device))
            Z  = simulate_one_gbm(batch_size, z0, sigma)
            feat = feat_inner(Z)
            g_call = payoff_call_unit(Z)
            g_put  = payoff_put_unit(Z)

        for lam in lambdas:
            inner_call[lam].train_one_batch_backward(feat, g_call)
            inner_put[lam].train_one_batch_backward(feat, g_put)

        if it == 1 or it % eval_every == 0:
            for lam in lambdas:
                v1, _ = inner_call[lam].evaluate_boundary_value(feat_test_call, g_test_call)
                v2, _ = inner_put[lam].evaluate_boundary_value(feat_test_put, g_test_put)
                hist_u1[lam].append((it, v1))
                hist_u2[lam].append((it, v2))

            msg = " | ".join(
                [f"λ={lam:g} u1={hist_u1[lam][-1][1]:.4f} u2={hist_u2[lam][-1][1]:.4f}"
                 for lam in lambdas]
            )
            print(f"\r[u] it={it:4d}/{iterations_inner} :: {msg}", end="")
            sys.stdout.flush()
    print()

    # Outer problem
    with torch.no_grad():
        for lam in lambdas:
            phi_test_all[lam], _, _ = phi_on_paths(
                X1_test, X2_test, inner_call[lam], inner_put[lam]
            )

    for it in range(1, iterations_outer + 1):
        with torch.no_grad():
            X1, X2 = simulate_two_gbm(batch_size)

        for lam in lambdas:
            phi_train, _, _ = phi_on_paths(X1, X2, inner_call[lam], inner_put[lam])
            feat_o = feat_outer(X1, X2)
            outer[lam].train_one_batch_backward(feat_o, phi_train)

        if it == 1 or it % eval_every == 0:
            for lam in lambdas:
                val, _ = outer[lam].evaluate_boundary_value(
                    feat_outer_test, phi_test_all[lam]
                )
                hist_phi[lam].append((it, val))

            msg = " | ".join(
                [f"λ={lam:g} φ={hist_phi[lam][-1][1]:.4f}"
                 for lam in lambdas]
            )
            print(f"\r[φ] it={it:4d}/{iterations_outer} :: {msg}", end="")
            sys.stdout.flush()
    print()

    for lam in lambdas:
        with torch.no_grad():
            ex_val, ex_sd = evaluate_exchange_option(
                X1_test, X2_test,
                inner_call[lam], inner_put[lam], outer[lam]
            )

        print(f"λ={lam:g} | Exchange value={ex_val:.6f} (sd={ex_sd:.6f})")

        pd.DataFrame(hist_u1[lam], columns=["iteration", "value"]).to_csv(
            f"Results/exchange_u1_lambda_{lam}_seed_{seed}.csv", index=False
        )
        pd.DataFrame(hist_u2[lam], columns=["iteration", "value"]).to_csv(
            f"Results/exchange_u2_lambda_{lam}_seed_{seed}.csv", index=False
        )
        pd.DataFrame(hist_phi[lam], columns=["iteration", "value"]).to_csv(
            f"Results/exchange_phi_lambda_{lam}_seed_{seed}.csv", index=False
        )
        pd.DataFrame([{
            "seed": seed, "lambda": lam,
            "exchange_value": ex_val, "exchange_sd": ex_sd
        }]).to_csv(
            f"Results/exchange_final_lambda_{lam}_seed_{seed}.csv", index=False
        )


print("Device:", device)
for seed in seeds:
    train_exchange_for_seed(seed)

In [ ]:
# Results table + plots
files = glob.glob(r"Results/exchange_phi_lambda_*_seed_*.csv")
data_by_lambda = defaultdict(list)
pat = re.compile(r"Results\\exchange_phi_lambda_(.*)_seed_(\d+)\.csv")

for fn in files:
    m = pat.match(os.path.join("Results",os.path.basename(fn)))
    if not m:
        continue
    lam = float(m.group(1))
    df = pd.read_csv(fn)
    data_by_lambda[lam].append(df)

# Results table
print("\nSummary over seeds (outer value u(0)):")
for lam in sorted(data_by_lambda.keys()):
    vals = np.array([df["value"].iloc[-1] for df in data_by_lambda[lam]], dtype=float)

    print(
        f"lambda = {lam:g} | "
        f"Value = {vals.mean():.6f} | "
        f"SD = {vals.std(ddof=1):.6f}"
    )

    
# Plots
def load_curves(pattern, value_col="value"):
    files = glob.glob(pattern)
    data = defaultdict(list)

    pat = re.compile(r"Results\\.*lambda_(.*)_seed_(\d+)\.csv")

    for fn in files:
        m = pat.match(fn)
        if not m:
            continue
        lam = float(m.group(1))
        df = pd.read_csv(fn)
        data[lam].append(df)

    return data

data_u1 = load_curves("Results/exchange_u1_lambda_*_seed_*.csv")

plt.figure(figsize=(8,5))
i = 0
for lam, dfs in reversed(sorted(data_u1.items())):
    vals = np.stack([df["value"].values for df in dfs])
    it = dfs[0]["iteration"].values
    mean = vals.mean(axis=0)
    std  = vals.std(axis=0)
    plt.plot(it, mean, color=COLOURS[i], label=fr"$\lambda={lam}$")
    i += 1

plt.xlabel("Number of Steps")
plt.ylabel(r"$u_1$ estimate")
plt.grid(alpha=0.3)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("Plots/exchange_u1_learning.png")
plt.show()


data_u2 = load_curves("Results/exchange_u2_lambda_*_seed_*.csv")

plt.figure(figsize=(8,5))
i = 0
for lam, dfs in reversed(sorted(data_u2.items())):
    vals = np.stack([df["value"].values for df in dfs])
    it = dfs[0]["iteration"].values
    mean = vals.mean(axis=0)
    plt.plot(it, mean, color=COLOURS[i], label=fr"$\lambda={lam}$")
    i += 1

plt.xlabel("Number of Steps")
plt.ylabel(r"$u_2$ estimate")
plt.grid(alpha=0.3)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("Plots/exchange_u2_learning.png")
plt.show()

data_phi = load_curves("Results/exchange_phi_lambda_*_seed_*.csv")

plt.figure(figsize=(8,5))
i = 0
for lam, dfs in reversed(sorted(data_phi.items())):
    vals = np.stack([df["value"].values for df in dfs])
    it = dfs[0]["iteration"].values
    mean = vals.mean(axis=0)
    plt.plot(it, mean, color=COLOURS[i], label=fr"$\lambda={lam}$")
    i += 1

plt.xlabel("Number of Steps")
plt.ylabel(r"$u$ estimate")
plt.grid(alpha=0.3)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("Plots/exchange_phi_learning.png")
plt.show()

## Experiment 3: American Put Option (Time-Inconsistent)
In this block, we implement the equilibrium-style iteration for time‑inconsistent stopping presented in Algorithm 4 by training value nets and updating the implied stopping rule across iterations.

In [ ]:
# Experiment 3: American Put Option (Time-Inconsistent)

S0     = 40.0
K      = 40.0
r      = 0.06
sigma  = 0.40
T      = 1.0

def payoff_put(S: torch.Tensor):
    return torch.clamp(K - S, min=0.0)

# Hyperbolic discount
def delta_t(t: torch.Tensor):
    return 1.0 / (1.0 + r * t)

L  = 50
dt = T / L

K_rate       = 25
assert K_rate * dt <= 1.0

EPS_STOP     = 0.5

iterations     = 3000
eval_steps     = 1
batch_size     = 2**10
test_size      = 2**18
learning_rate  = 1e-3
eval_every     = 10

seeds = [i for i in range(1, 101)]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

os.makedirs("Results", exist_ok=True)
os.makedirs("Plots", exist_ok=True)


# Simulated GBM data
def simulate_gbm(N: int):
    Z = torch.randn((N, L), device=device)
    drift = (r - 0.5 * sigma**2) * dt
    vol = sigma * math.sqrt(dt)

    log_increments = drift + vol * Z

    logS = torch.empty((N, L+1), device=device)
    logS[:, 0] = math.log(S0)
    logS[:, 1:] = logS[:, [0]] + torch.cumsum(log_increments, dim=1)

    return torch.exp(logS)  # (N, L+1)


# Time-slice value network
class TimeSliceNet(nn.Module):
    def __init__(self, width=21):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, width), nn.ReLU(),
            nn.Linear(width, width), nn.ReLU(),
            nn.Linear(width, 1)
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, S, g, t_norm):
        x = torch.stack([S, g, t_norm], dim=-1)
        return g + self.net(x).squeeze(-1)


# Policy Update
@torch.no_grad()
def alpha_from_prev_V0(S, V0_prev_net):
    if V0_prev_net is None:
        return torch.full_like(S, K_rate)

    g = payoff_put(S)
    t0 = torch.zeros_like(S)
    V0 = V0_prev_net(S, g, t0)

    g_eps = 1e-8
    mask = (g >= V0 - EPS_STOP) & (g > g_eps)
    return torch.where(mask, torch.full_like(S, K_rate), torch.zeros_like(S))


# Boundary evaluation
@torch.no_grad()
def evaluate_boundary(nets, S_test):
    g_test = payoff_put(S_test)
    times  = torch.arange(L+1, device=device) * dt
    disc   = delta_t(times)

    payoff    = torch.zeros(S_test.shape[0], device=device)
    exercised = torch.zeros(S_test.shape[0], dtype=torch.bool, device=device)

    for l in range(1, L):
        if exercised.all():
            break

        S_l = S_test[:, l]
        g_l = g_test[:, l]
        t_norm = torch.full_like(S_l, l / L)

        V_l = nets[l](S_l, g_l, t_norm)

        immediate = disc[l] * g_l
        stop = (V_l <= immediate - 1e-6) & (~exercised)

        payoff[stop] = immediate[stop]
        exercised[stop] = True

    payoff[~exercised] = disc[L] * g_test[~exercised, L]
    return payoff.mean().item()


# Train Algorithm 4
def train_algorithm4_finite(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    nets = nn.ModuleList([TimeSliceNet().to(device) for _ in range(L)])
    opts = [optim.Adam(net.parameters(), lr=learning_rate) for net in nets]

    use_amp = (device.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    with torch.no_grad():
        S_test = simulate_gbm(test_size)

    times  = torch.arange(L+1, device=device) * dt
    disc   = delta_t(times)  # (L+1,)
    t_norms = (torch.arange(L+1, device=device, dtype=torch.float32) / L)

    V0_prev_net = None
    history = []

    for n in range(1, iterations + 1):

        # Policy Evaluation
        for _ in range(eval_steps):
            S = simulate_gbm(batch_size)
            g = payoff_put(S)

            V_terminal = disc[L] * g[:, L]

            for l in reversed(range(L)):
                S_l = S[:, l]
                g_l = g[:, l]
                t_norm = torch.full_like(S_l, t_norms[l].item())

                with torch.no_grad():
                    a_l = alpha_from_prev_V0(S_l, V0_prev_net)

                    if l == L - 1:
                        V_next = V_terminal
                    else:
                        S_next = S[:, l+1]
                        g_next = g[:, l+1]
                        t_next = torch.full_like(S_l, t_norms[l+1].item())
                        V_next = nets[l+1](S_next, g_next, t_next)

                opt = opts[l]
                opt.zero_grad(set_to_none=True)

                with torch.cuda.amp.autocast(enabled=use_amp):
                    V_l = nets[l](S_l, g_l, t_norm)
                    td = disc[l] * g_l * a_l * dt + (1.0 - a_l * dt) * V_next - V_l
                    loss = (td ** 2).mean()

                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()

        # Policy Update
        V0_prev_net = TimeSliceNet().to(device)
        V0_prev_net.load_state_dict(nets[0].state_dict())
        V0_prev_net.eval()

        # Evaluation
        if (n == 1) or (n % eval_every == 0):
            val = evaluate_boundary(nets, S_test)
            history.append((n, val))
            print(f"\rSeed {seed} | iter {n:4d}/{iterations} | boundary value={val:.6f}", end="")
            sys.stdout.flush()

    print()
    return np.array(history, dtype=float)


# Run + save
for seed in seeds:
    hist = train_algorithm4_finite(seed)
    df = pd.DataFrame(hist, columns=["iteration", "value"])
    df.to_csv(f"Results/inconsistent_put_alg4_finite_T{T:g}_K{int(K_rate)}_seed_{seed}.csv", index=False)


In [ ]:
# Results Table + Plot
Ks = [4, 10, 25]

results_dir = Path("Results")
if not results_dir.exists():
    candidates = list(Path.cwd().rglob("Results"))
    candidates = [p for p in candidates if p.is_dir()]
    if len(candidates) == 0:
        raise FileNotFoundError(
            "Could not find a 'Results' folder. "
            "Run this cell from the project root, or set results_dir explicitly."
        )
    results_dir = candidates[0]

files = sorted(results_dir.glob("inconsistent_put_alg4_finite_T1_K*_seed_*.csv"))

data_by_K = defaultdict(list)

pattern = re.compile(r".*Results[\\/]+inconsistent_put_alg4_finite_T1_K(\d+)_seed_(\d+)\.csv$")

for f in files:
    m = pattern.match(str(f))
    if m is None:
        continue
    K_str, seed_str = m.groups()
    K_val = int(K_str)
    if K_val in Ks:
        df = pd.read_csv(f)
        data_by_K[K_val].append(df)

# Result table
for K_val in Ks:
    dfs = data_by_K.get(K_val, [])
    if len(dfs) == 0:
        continue
    finals = [df["value"].iloc[-1] for df in dfs]
    print(f"K={K_val:>2d} | Value={np.mean(finals):.4f} | SD={np.std(finals, ddof=1):.3f} | n={len(finals)}")


# Plot        
plt.figure(figsize=(8, 5))

i = 0
for K_val in Ks:
    dfs = data_by_K.get(K_val, [])
    values = np.stack([df["value"].values for df in dfs], axis=0)
    iters  = dfs[0]["iteration"].values
    mean_vals = values.mean(axis=0)
    plt.plot(iters, mean_vals, color=COLOURS[i], label=fr"K = {K_val}")
    i += 1

plt.xlabel("Number of Steps")
plt.ylabel("Value at t=0")
plt.grid(alpha=0.3)

plt.legend()
plt.tight_layout()
plt.savefig("Plots/RL_put_inconsistent_alg4_finite_T1_multiK.png")
plt.show()